# Kapampangan MorphBPE: Colab reproduction

CPU-only reproduction of tokenizer training and validation-only selection. Upload/extract this project and the immutable portable dataset into `/content`, or mount Drive and set the two environment variables below. The notebook never reads `data/test.csv`, never downloads NLLB, and never runs final held-out evaluation.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("KAPAMPANGAN_MORPHBPE_PROJECT", "/content/kapampangan-morphbpe-paper-v1")).resolve()
DATASET_ROOT = Path(os.environ.get("KAPAMPANGAN_DATASET_ROOT", "/content/kapampangan-general-corpus-v1")).resolve()
assert (PROJECT_ROOT / "pyproject.toml").is_file(), "Upload or mount the project, then set KAPAMPANGAN_MORPHBPE_PROJECT"
assert (DATASET_ROOT / "metadata/dataset-manifest.json").is_file(), "Upload/extract or mount the portable dataset, then set KAPAMPANGAN_DATASET_ROOT"
print(PROJECT_ROOT)
print(DATASET_ROOT)

## Build the pinned Python/Rust environment

Maturin builds the local PyO3 extension. Colab's CPU runtime is sufficient; no GPU package is installed.

In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-lock.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", str(PROJECT_ROOT)], check=True)

## Verify inputs and build the training-only lexicon

The verifier requires corpus fingerprint `aff5de7b8fc158f144eaec4af3e3c22faa3b184859925130a04604a2aa9c5d17` and the portable ZIP/package file hashes. Test bytes may be checksum-verified, but test rows are never parsed.

In [ ]:
def run(*arguments: str) -> None:
    subprocess.run([sys.executable, "-m", "kapampangan_morphbpe", *arguments], cwd=PROJECT_ROOT, check=True)

run("verify-dataset", "--dataset-root", str(DATASET_ROOT), "--output", str(PROJECT_ROOT / "reports/dataset-verification-colab.json"))
run("build-lexicon", "--dataset-root", str(DATASET_ROOT), "--output-dir", str(PROJECT_ROOT / "resources"))
run("verify-segmentation-parity", "--dataset-root", str(DATASET_ROOT), "--lexicon", str(PROJECT_ROOT / "resources/training-lexicon.json"), "--output", str(PROJECT_ROOT / "reports/segmentation-parity-colab.json"))

## Prepare, freeze, and train candidates

The Rust segmenter creates the train-only stream. Candidate sizes are frozen before validation evaluation.

In [ ]:
prepared = PROJECT_ROOT / "runs/prepared"
grid = PROJECT_ROOT / "configs/candidate-grid.json"
run("prepare-training", "--dataset-root", str(DATASET_ROOT), "--lexicon", str(PROJECT_ROOT / "resources/training-lexicon.json"), "--output-dir", str(prepared), "--engine", "rust")
run("freeze-candidates", "--prepared-manifest", str(prepared / "training-stream-manifest.json"), "--output", str(grid))
run("train-candidates", "--prepared-stream", str(prepared / "training-stream.jsonl"), "--prepared-manifest", str(prepared / "training-stream-manifest.json"), "--grid", str(grid), "--lexicon-manifest", str(PROJECT_ROOT / "resources/training-lexicon-manifest.json"), "--output-dir", str(PROJECT_ROOT / "artifacts/candidates"))

## Validation-only selection and deterministic finalization

In [ ]:
validation_reports = PROJECT_ROOT / "reports/validation"
selection = PROJECT_ROOT / "configs/selected-candidate.json"
run("evaluate-validation", "--dataset-root", str(DATASET_ROOT), "--lexicon", str(PROJECT_ROOT / "resources/training-lexicon.json"), "--candidates-dir", str(PROJECT_ROOT / "artifacts/candidates"), "--grid", str(grid), "--reports-dir", str(validation_reports))
run("select-candidate", "--grid", str(grid), "--validation-report", str(validation_reports / "validation-candidates.json"), "--output", str(selection))
run("finalize-tokenizer", "--prepared-stream", str(prepared / "training-stream.jsonl"), "--selection", str(selection), "--lexicon-manifest", str(PROJECT_ROOT / "resources/training-lexicon-manifest.json"), "--prepared-manifest", str(prepared / "training-stream-manifest.json"), "--output-dir", str(PROJECT_ROOT / "artifacts/selected-tokenizer"), "--rebuild-dir", str(PROJECT_ROOT / "artifacts/determinism-rebuild"))

## Validate/export the tokenizer artifact

This exports only the future NLLB adapter contract. It does not install, download, or train NLLB.

In [ ]:
import shutil

selected = PROJECT_ROOT / "artifacts/selected-tokenizer"
run("validate-artifact", "--artifact", str(selected))
run("verify-runtime-independence", "--artifact", str(selected), "--runtime-root", str(PROJECT_ROOT / "runtime"), "--output", str(PROJECT_ROOT / "reports/runtime-independence-colab.json"))
run("export-nllb-contract", "--artifact", str(selected), "--output-dir", str(PROJECT_ROOT / "nllb"))
archive = shutil.make_archive(str(PROJECT_ROOT / "artifacts/selected-tokenizer"), "zip", root_dir=selected)
print(archive)